# ROBIN Watermark Benchmark — Google Colab
**Generation-based (SD2.1 + optimised latent patch).**  
Runs 14 WAVES attacks, reports PSNR / SSIM / AUROC / TPR@1% FPR.

### Setup
1. Runtime → **GPU** (A100 recommended; T4 works but slower)
2. Run all cells — clones code from [github.com/ademladhari/wavess](https://github.com/ademladhari/wavess)
3. Optional: mount Drive in cell 1 to save results to `MyDrive/wavess_results/`

In [ ]:
# ── 1. Clone wavess + optional Drive mount ───────────────────────────────────
import os, sys, subprocess

REPO_URL   = 'https://github.com/ademladhari/wavess.git'
WAVES_ROOT = '/content/wavess'

if not os.path.isdir(f'{WAVES_ROOT}/ROBIN'):
    print('Cloning wavess (with ssl submodule)…')
    subprocess.run(['git', 'clone', '--depth', '1', '--recurse-submodules', REPO_URL, WAVES_ROOT], check=True)
else:
    print('Repo already present:', WAVES_ROOT)

ROBIN_ROOT = f'{WAVES_ROOT}/ROBIN'
for p in [WAVES_ROOT, ROBIN_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(ROBIN_ROOT)
os.makedirs('logs', exist_ok=True)

# Optional: mount Drive to persist results (set SAVE_TO_DRIVE = True)
SAVE_TO_DRIVE = True
DRIVE_OUTPUT  = '/content/drive/MyDrive/wavess_results/robin'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Shared attack suite — must live at repo root (not inside ROBIN/)
_ba = f'{WAVES_ROOT}/benchmark_attacks.py'
if not os.path.isfile(_ba):
    raise FileNotFoundError(
        f'Missing {_ba}\n'
        'Push benchmark_attacks.py to GitHub, then re-run this cell:\n'
        '  cd D:\\waves\n'
        '  git push origin main\n'
        'Or pull latest: rm -rf /content/wavess and re-clone.'
    )
print('Repo OK:', WAVES_ROOT, '| benchmark_attacks.py found')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q diffusers==0.38.0 transformers accelerate pytorch-msssim \
               scikit-image scikit-learn tqdm

In [ ]:
# ── 3. Config — edit here ────────────────────────────────────────────────────
N_IMAGES         = 100          # total watermarked images to generate
BATCH_SIZE       = 4            # images generated per SD forward pass
DETECT_BATCH     = 8            # images inverted per batch (memory permitting)
MODEL_ID         = 'sd2-community/stable-diffusion-2-1-base'
WM_PATH          = None         # None → auto-find latest .pt in ROBIN/ckpts/
PROMPTS_FILE     = f'{WAVES_ROOT}/tree-ring/prompts.txt'
OUTPUT_DIR       = f'{WAVES_ROOT}/ROBIN/outputs_benchmark_colab'  # local fast disk
SEED             = 42
NUM_STEPS        = 50
WATERMARK_STEPS  = 35
DETECT_STEPS     = 50
GUIDANCE         = 7.5
IMAGE_LEN        = 512
DEVICE           = 'cuda'

In [ ]:
# ── 4. Imports ────────────────────────────────────────────────────────────────
import copy, csv, json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from diffusers import DPMSolverMultistepScheduler
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn import metrics as sk_metrics
from tqdm.auto import tqdm

from benchmark_attacks import ATTACKS, apply_attack_rgb
from inverse_stable_diffusion import InversableStableDiffusionPipeline
from optim_utils import get_watermarking_mask, set_random_seed, transform_img

print('Imports OK  |  CUDA:', torch.cuda.is_available(),
      '|  GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# ── 5. Helpers ────────────────────────────────────────────────────────────────
def load_prompts(path, n):
    lines = [l.strip() for l in Path(path).read_text('utf-8').splitlines() if l.strip()]
    if len(lines) < n:
        lines = (lines * ((n // len(lines)) + 1))[:n]
    return lines[:n]

def resolve_wm(path):
    if path and Path(path).is_file(): return Path(path)
    for base in (Path(ROBIN_ROOT)/'ckpts', Path(ROBIN_ROOT)):
        pts = sorted(base.rglob('*.pt')) if base.is_dir() else []
        if pts: return pts[-1]
    raise FileNotFoundError('No .pt checkpoint found. Set WM_PATH.')

def psnr_ssim(ref, atk):
    b = atk.resize(ref.size, Image.Resampling.LANCZOS) if atk.size != ref.size else atk
    a = np.asarray(ref, dtype=np.float64)
    b = np.asarray(b,   dtype=np.float64)
    return (float(peak_signal_noise_ratio(a, b, data_range=255.0)),
            float(structural_similarity(a, b, channel_axis=2, data_range=255.0)))

def auroc_tpr(pos, neg, fpr=0.01):
    y = np.concatenate([np.zeros(neg.size, dtype=np.int32), np.ones(pos.size, dtype=np.int32)])
    s = np.concatenate([neg, pos])
    auc = float(sk_metrics.roc_auc_score(y, s))
    fp, tp, _ = sk_metrics.roc_curve(y, s, pos_label=1)
    idx = np.where(fp < fpr)[0]
    return auc, float(tp[idx[-1]]) if idx.size else float(tp[0])

def build_args():
    return SimpleNamespace(
        w_seed=999999, w_channel=3, w_pattern='ring', w_mask_shape='circle',
        w_up_radius=30, w_low_radius=5, w_measurement='l1_complex',
        w_injection='complex', w_pattern_const=0.0,
    )

print('Helpers defined.')

In [ ]:
# ── 6. Load model & checkpoint ───────────────────────────────────────────────
wm_file = resolve_wm(WM_PATH)
print(f'Checkpoint: {wm_file}')

scheduler = DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')
pipe = InversableStableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    scheduler=scheduler,
    torch_dtype=torch.float32,
    safety_checker=None,
    feature_extractor=None,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

ckpt = torch.load(wm_file, map_location='cpu')
opt_wm   = ckpt['opt_wm'].to(DEVICE)
opt_acond = ckpt.get('opt_acond')
text_emb  = pipe.get_text_embedding('')
if opt_acond is not None:
    opt_acond = opt_acond.to(text_emb.dtype).to(DEVICE)

wm_args   = build_args()
init_lat  = pipe.get_random_latents(height=IMAGE_LEN, width=IMAGE_LEN)
wm_mask   = get_watermarking_mask(init_lat, wm_args, DEVICE)

print('Model loaded.')

In [ ]:
# ── 7. Generate watermarked + clean images (per-image progress) ────────────────
from time import time

prompts = load_prompts(PROMPTS_FILE, N_IMAGES)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

records = []

def get_latents(seed):
    set_random_seed(seed)
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    return pipe.get_random_latents(height=IMAGE_LEN, width=IMAGE_LEN, generator=gen)

print(f'Generating {N_IMAGES} image pairs (wm + clean)…')
t0 = time()
for i in tqdm(range(N_IMAGES), desc='images', unit='img'):
    gen_seed = SEED + i
    text     = prompts[i]
    lat      = get_latents(gen_seed)

    wm_out = pipe(
        text, num_images_per_prompt=1, guidance_scale=GUIDANCE,
        num_inference_steps=NUM_STEPS, height=IMAGE_LEN, width=IMAGE_LEN,
        latents=copy.deepcopy(lat),
        watermarking_mask=wm_mask, watermarking_steps=WATERMARK_STEPS,
        args=wm_args, gt_patch=opt_wm, lguidance=GUIDANCE, opt_acond=opt_acond,
    )
    clean_out = pipe(
        text, num_images_per_prompt=1, guidance_scale=GUIDANCE,
        num_inference_steps=NUM_STEPS, height=IMAGE_LEN, width=IMAGE_LEN,
        latents=get_latents(gen_seed),
    )
    records.append({
        'prompt': text,
        'wm_pil': wm_out.images[0].convert('RGB'),
        'clean_pil': clean_out.images[0].convert('RGB'),
        'seed': gen_seed,
    })

    if (i + 1) % 10 == 0:
        elapsed = time() - t0
        rate = (i + 1) / elapsed
        eta_h = (N_IMAGES - i - 1) / rate / 3600
        print(f'  [{i+1}/{N_IMAGES}] {rate:.3f} img/s — ETA {eta_h:.1f} h', flush=True)

print(f'Generated {len(records)} pairs in {(time()-t0)/3600:.2f} h')

In [ ]:
# ── 8. Detection (batched DDIM inversion) ────────────────────────────────────
def detect_batch(images):
    """Score a list of PIL images; returns list of float scores."""
    scores = []
    for img in images:   # inversion is inherently sequential
        im_t = (transform_img(img, target_size=IMAGE_LEN)
                .unsqueeze(0).to(text_emb.dtype).to(DEVICE))
        lat_img = pipe.get_image_latents(im_t, sample=False)
        rev = pipe.forward_diffusion(
            latents=lat_img, text_embeddings=text_emb,
            guidance_scale=1.0, num_inference_steps=DETECT_STEPS,
        )
        fft = torch.fft.fftshift(torch.fft.fft2(rev), dim=(-1, -2))
        dist = torch.abs(fft[wm_mask] - opt_wm.to(fft.dtype)[wm_mask]).mean().item()
        scores.append(float(np.clip(1.0 / (1.0 + dist), 0.0, 1.0)))
    return scores

# sanity check
sanity = detect_batch([records[0]['wm_pil']])[0]
print(f'Sanity (image 0, no attack): detect_score = {sanity:.4f}')

# negative scores (clean images)
print('Scoring negatives…')
neg_scores = []
for i in tqdm(range(0, len(records), DETECT_BATCH), desc='negatives'):
    batch_imgs = [r['clean_pil'] for r in records[i:i+DETECT_BATCH]]
    neg_scores.extend(detect_batch(batch_imgs))
neg_arr = np.asarray(neg_scores, dtype=np.float64)
print(f'Neg score mean: {neg_arr.mean():.4f}')

In [ ]:
# ── 9. Per-attack evaluation ──────────────────────────────────────────────────
rows = []
for spec in ATTACKS:
    psnr_vals, ssim_vals, pos_scores = [], [], []

    attacked_imgs = [apply_attack_rgb(spec.name, r['wm_pil'], seed=i)
                     for i, r in enumerate(records)]

    for ref, atk in zip(records, attacked_imgs):
        p, s = psnr_ssim(ref['wm_pil'], atk)
        psnr_vals.append(p)
        ssim_vals.append(s)

    # detect in batches
    for i in range(0, len(attacked_imgs), DETECT_BATCH):
        pos_scores.extend(detect_batch(attacked_imgs[i:i+DETECT_BATCH]))

    pos_arr = np.asarray(pos_scores, dtype=np.float64)
    auc, tpr1 = auroc_tpr(pos_arr, neg_arr)
    row = {
        'method': 'robin', 'attack': spec.name, 'description': spec.description,
        'n_images': len(records),
        'PSNR': float(np.mean(psnr_vals)), 'SSIM': float(np.mean(ssim_vals)),
        'bit_accuracy': float('nan'), 'AUROC': auc, 'TPR_at_1pct_FPR': tpr1,
    }
    rows.append(row)
    print(f"  {spec.name:22s}: PSNR={row['PSNR']:5.1f}  AUROC={auc:.3f}  TPR@1%={tpr1:.3f}")

print('\nAll attacks done.')

In [ ]:
# ── 10. Save results ──────────────────────────────────────────────────────────
import csv as _csv
csv_path  = Path(OUTPUT_DIR) / 'results.csv'
json_path = Path(OUTPUT_DIR) / 'results.json'

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = _csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({'method': 'robin', 'n_images': len(records),
               'model_id': MODEL_ID, 'wm_path': str(wm_file),
               'sanity_score': sanity, 'attacks': rows}, f, indent=2)

print(f'Results saved to {OUTPUT_DIR}')

# sync to Drive if mounted
if SAVE_TO_DRIVE:
    import shutil
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'Copied to Drive: {DRIVE_OUTPUT}')

import pandas as pd
df = pd.DataFrame(rows)[['attack','PSNR','SSIM','AUROC','TPR_at_1pct_FPR']]
df.columns = ['Attack','PSNR','SSIM','AUROC','TPR@1%FPR']
print(df.to_string(index=False))